In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "4,5,6,7"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader

/mnt/petrelfs/zhangshilin/anaconda3/envs/deepscaler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-26 17:27:54,767	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
model_path = "/mnt/petrelfs/zhangshilin/rlvr_div/checkpoints/div/constrastive_clp_028_t/actor/global_step_100"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).cuda()

Loading checkpoint shards: 100%|██████████| 7/7 [00:02<00:00,  2.51it/s]


In [3]:
train_data_path = "/mnt/petrelfs/zhangshilin/rlvr_div/dataset/valid.all.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=3000,
                            filter_prompts=False,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=1,
                            shuffle=False,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 6023
filter dataset len: 6023


In [18]:
# prompt = "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$"
prompt = "In how many ways can $7$ people sit around a round table if no two of the $3$ people Pierre, Rosa, and Thomas can sit next to each other? (Seating arrangements which are rotations of each other are treated as the same.)" # 144
messages = [
    {"role": "system", "content": "Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\n {thoughts} </think>\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions. Assistant: <think>\n"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
num_samples = 3

batch_generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=5000,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    num_return_sequences=num_samples,  # 一次生成3个不同的序列
)

# 处理一次性生成的多个回答
batch_responses = []
for i in range(num_samples):
    # 提取新生成的token (需要注意input_ids复制了3次)
    gen_ids = batch_generated_ids[i][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    batch_responses.append(response)
    
    print(f"批量生成 - 回答 {i+1}:")
    print(response)
    print("\n" + "="*50 + "\n")

# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=512
# )
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


批量生成 - 回答 1:
To solve this problem, we need to find the number of ways to arrange 7 people around a round table such that no two of Pierre (P), Rosa (R), and Thomas (T) sit next to each other. In a circular arrangement, the total number of ways to arrange 7 people is \((7-1)!\) (i.e., \(6!\)), which equals 720. We need to subtract the number of arrangements where at least two of Pierre, Rosa, and Thomas are seated next to each other.

First, let's find the total number of arrangements where Pierre, Rosa, and Thomas can sit next to each other. We can treat each block of two people sitting together as a single "super person". For each case where two of Pierre, Rosa, and Thomas sit together, we have:

1. \(P\) and \(R\) sit together, \(T\) is separate.
2. \(P\) and \(T\) sit together, \(R\) is separate.
3. \(R\) and \(T\) sit together, \(P\) is separate.
4. \(P\), \(R\), and \(T\) all sit together.

Let's calculate each of these cases:

1. If \(P\) and \(R\) sit together, we treat \(PR\) 

In [4]:
def get_fixed_sample(dataset, index=0):
    # 确保索引在有效范围内
    if index >= len(dataset):
        index = 0
    return dataset[index:index+1]  # 获取固定的3个样本(按batch_size=3)

fixed_sample = get_fixed_sample(train_dataset, index=0)  # 可以改变index来选择不同的样本
test_data = collate_fn([fixed_sample])  # 应用collate_fn处理数据

print(test_data.keys())
seq = tokenizer.batch_decode(test_data['input_ids'], skip_special_tokens=True)
print(seq)
# 准备输入
input_ids = test_data['input_ids'].to(model.device)
print(input_ids.shape)

dict_keys(['input_ids', 'attention_mask', 'position_ids', 'data_source', 'ability', 'reward_model', 'extra_info', 'index'])
['Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\n {thoughts} </think>\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions. Assistant: <think>\n']
torch.Size([1, 3000])


In [9]:
import pandas as pd

# 直接读取parquet文件中的数据
df = pd.read_parquet("/mnt/petrelfs/zhangshilin/rlvr_div/dataset/valid.all.parquet")
# 选择第一行并打印所有列
print(df.iloc[0].to_dict())

# # 假设题目在'prompt'列中
# if 'prompt' in df.columns:
#     prompt = df['prompt'].iloc[0]
#     print("原始题目:", prompt)

{'data_source': 'math', 'prompt': array([{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.', 'role': 'system'},
       {'content': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and

In [6]:
print('=======')
attention_mask = test_data['attention_mask'].to(model.device)
group_rollout = []
for i in range(4):
    # 生成文本
    with torch.no_grad():
        gene = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=8192,  # 生成新token的数量
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            # repetition_penalty=1.2  # 避免重复
        )
    
    # 解码生成的序列
    original_length = input_ids.shape[1]
    print(original_length)
    new_tokens = gene[:, original_length:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    group_rollout.append(generated_texts)

for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])
    print("******************")

3000
3000
3000
3000
********0**********
["The problem asks us to derive the value of \\(x\\) given the system of equations. However, there are no equations provided in the prompt. Normally, the value of \\(x\\) would be solved by analyzing and solving the equations. For this example, let's assume we have the following system of linear equations as an example:\n\n\\[\n\\begin{cases}\n2x + 3y = 7 \\\\\nx - 2y = 1\n\\end{cases}\n\\]\n\nFirst, we will solve the second equation for \\(x\\):\n\\[\nx = 1 + 2y\n\\]\n\nNext, substitute this expression for \\(x\\) into the first equation:\n\\[\n2(1 + 2y) + 3y = 7\n\\]\n\nThis simplifies to:\n\\[\n2 + 4y + 3y = 7 \\implies 2 + 7y = 7 \\implies 7y = 5 \\implies y = \\frac{5}{7}\n\\]\n\nNow, substitute \\(y = \\frac{5}{7}\\) back into the equation \\(x = 1 + 2y\\):\n\\[\nx = 1 + 2 \\left(\\frac{5}{7}\\right) = 1 + \\frac{10}{7} = \\frac{7}{7} + \\frac{10}{7} = \\frac{17}{7}\n\\]\n\n</think>\n\nBased on the reasoning and solving the equations, the v

In [7]:
test = torch.tensor([[1, 1, 1], [1, 2, 2], [1, 3, 3],
                    [2, 1, 1], [2, 2, 2], [2, 3, 3],
                    [3, 1, 1], [3, 2, 2], [3, 3, 3],
                    [4, 1, 1], [4, 2, 2], [4, 3, 3]])
bsz, hidden_dim = test.shape
grouped = test.reshape(-1, 3, hidden_dim)
print(grouped)

tensor([[[1, 1, 1],
         [1, 2, 2],
         [1, 3, 3]],

        [[2, 1, 1],
         [2, 2, 2],
         [2, 3, 3]],

        [[3, 1, 1],
         [3, 2, 2],
         [3, 3, 3]],

        [[4, 1, 1],
         [4, 2, 2],
         [4, 3, 3]]])


In [8]:
grouped.repeat_interleave(3, dim=0).reshape(bsz,3,hidden_dim)

tensor([[[1, 1, 1],
         [1, 2, 2],
         [1, 3, 3]],

        [[1, 1, 1],
         [1, 2, 2],
         [1, 3, 3]],

        [[1, 1, 1],
         [1, 2, 2],
         [1, 3, 3]],

        [[2, 1, 1],
         [2, 2, 2],
         [2, 3, 3]],

        [[2, 1, 1],
         [2, 2, 2],
         [2, 3, 3]],

        [[2, 1, 1],
         [2, 2, 2],
         [2, 3, 3]],

        [[3, 1, 1],
         [3, 2, 2],
         [3, 3, 3]],

        [[3, 1, 1],
         [3, 2, 2],
         [3, 3, 3]],

        [[3, 1, 1],
         [3, 2, 2],
         [3, 3, 3]],

        [[4, 1, 1],
         [4, 2, 2],
         [4, 3, 3]],

        [[4, 1, 1],
         [4, 2, 2],
         [4, 3, 3]],

        [[4, 1, 1],
         [4, 2, 2],
         [4, 3, 3]]])